In [1]:
ECB_URL = (
    "https://data-api.ecb.europa.eu/service/data/EXR/D.USD.EUR.SP00.A"
    "?format=csvdata"
    "&startPeriod=2010-01-01"
    "&endPeriod=2024-12-31"
    "&detail=dataonly"        
)

StatementMeta(, 7bbf883a-1de0-49db-8f80-bc9d00581f3c, 3, Finished, Available, Finished, False)

In [2]:
import requests
import os
import io
from datetime import date, datetime

BRONZE_FILES = "/lakehouse/default/Files/fx"
os.makedirs(BRONZE_FILES, exist_ok=True)

StatementMeta(, 7bbf883a-1de0-49db-8f80-bc9d00581f3c, 4, Finished, Available, Finished, False)

In [3]:
print(f"\n{'='*60}")
print(f"  ECB FX Bronze Ingestion – USD/EUR daily rates")
print(f"  Started: {datetime.utcnow().isoformat()}Z")
print(f"{'='*60}\n")

print(f"  GET {ECB_URL[:80]}…")

headers = {"Accept": "text/csv"}
r = requests.get(ECB_URL, headers=headers, timeout=60)
r.raise_for_status()

raw_csv = r.text
lines   = raw_csv.strip().splitlines()
print(f"  [OK] Received {len(lines):,} lines  ({len(r.content)/1024:.1f} KB)")

StatementMeta(, 7bbf883a-1de0-49db-8f80-bc9d00581f3c, 5, Finished, Available, Finished, False)


  ECB FX Bronze Ingestion – USD/EUR daily rates
  Started: 2026-05-22T06:31:56.663956Z

  GET https://data-api.ecb.europa.eu/service/data/EXR/D.USD.EUR.SP00.A?format=csvdata&…
  [OK] Received 3,852 lines  (214.0 KB)


In [5]:
run_date  = date.today().isoformat()
out_file  = os.path.join(BRONZE_FILES, f"ecb_fx_usd_eur_{run_date}.csv")

with open(out_file, "w", encoding="utf-8") as f:
    f.write(raw_csv)

print(f"  [OK] Written: {out_file}")

for line in lines[:4]:
    print(f"    {line}")

StatementMeta(, 7bbf883a-1de0-49db-8f80-bc9d00581f3c, 7, Finished, Available, Finished, False)

  [OK] Written: /lakehouse/default/Files/fx/ecb_fx_usd_eur_2026-05-22.csv
    KEY,FREQ,CURRENCY,CURRENCY_DENOM,EXR_TYPE,EXR_SUFFIX,TIME_PERIOD,OBS_VALUE
    EXR.D.USD.EUR.SP00.A,D,USD,EUR,SP00,A,2010-01-01,
    EXR.D.USD.EUR.SP00.A,D,USD,EUR,SP00,A,2010-01-04,1.4389
    EXR.D.USD.EUR.SP00.A,D,USD,EUR,SP00,A,2010-01-05,1.4442


In [6]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import lit, current_timestamp, col, to_date
from pyspark.sql.types import DoubleType

spark = SparkSession.builder.getOrCreate()

raw_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(f"Files/fx/ecb_fx_usd_eur_{run_date}.csv")
    .withColumn("_ingested_at", current_timestamp())
    .withColumn("_source_file", lit(f"ecb_fx_usd_eur_{run_date}.csv"))
)

print("\nRaw schema:")
raw_df.printSchema()
print(f"Rows: {raw_df.count():,}")

if "TIME_PERIOD" in raw_df.columns and "OBS_VALUE" in raw_df.columns:
    raw_df = (
        raw_df
        .withColumnRenamed("TIME_PERIOD", "rate_date_str")
        .withColumnRenamed("OBS_VALUE",   "usd_eur_rate_raw")
    )

(
    raw_df.write
    .format("delta")
    .mode("overwrite")                  
    .option("overwriteSchema", "true")
    .saveAsTable("fx_raw")
)

print("\n  [OK] Delta table 'fx_raw' updated.")

StatementMeta(, 7bbf883a-1de0-49db-8f80-bc9d00581f3c, 8, Finished, Available, Finished, False)


Raw schema:
root
 |-- KEY: string (nullable = true)
 |-- FREQ: string (nullable = true)
 |-- CURRENCY: string (nullable = true)
 |-- CURRENCY_DENOM: string (nullable = true)
 |-- EXR_TYPE: string (nullable = true)
 |-- EXR_SUFFIX: string (nullable = true)
 |-- TIME_PERIOD: date (nullable = true)
 |-- OBS_VALUE: double (nullable = true)
 |-- _ingested_at: timestamp (nullable = false)
 |-- _source_file: string (nullable = false)

Rows: 3,851

  [OK] Delta table 'fx_raw' updated.


In [7]:
spark.sql("""
    SELECT
        MIN(rate_date_str) AS earliest,
        MAX(rate_date_str) AS latest,
        COUNT(*)           AS total_rows,
        ROUND(AVG(CAST(usd_eur_rate_raw AS DOUBLE)), 4) AS avg_rate
    FROM fx_raw
""").show()

StatementMeta(, 7bbf883a-1de0-49db-8f80-bc9d00581f3c, 9, Finished, Available, Finished, False)

+----------+----------+----------+--------+
|  earliest|    latest|total_rows|avg_rate|
+----------+----------+----------+--------+
|2010-01-01|2024-12-31|      3851|  1.1899|
+----------+----------+----------+--------+

